# 🧹 AgentCore End-to-End Cleanup

This notebook tears down every resource created across Labs 1-6.

## Overview

Unlike the original workshop, every resource in this project — IAM roles, ECR, S3, DynamoDB, Lambda, both Cognito pools, the AgentCore Gateway/Policy Engine, Knowledge Base, Memory, Runtime, Online Evaluation, and CloudWatch log delivery — is provisioned by **Terraform** (`iac/`), not by code running inside these notebooks. So cleanup is mostly a single command: `terraform destroy`.

Two AWS resource types Terraform manages here don't allow themselves to be deleted while non-empty (and don't have `force_destroy`/`force_delete` configured), so `terraform destroy` would otherwise fail partway through:

- **S3 buckets** (`artifacts`, `kb-data`) — contain the Lambda/layer zips and Knowledge Base source documents
- **ECR repository** — contains every container image pushed in Lab 4

This notebook empties both before handing off to `terraform destroy`, then verifies the stack is actually gone.

⚠️ **Important**: This cleanup is irreversible — it deletes the agent's Memory (conversation history), the Knowledge Base, and every other resource this project created. Make sure you're done experimenting before proceeding.

---

## Step 1: Locate the Resources to Clean Up

Read the S3 bucket and ECR repository names straight from Terraform's outputs — never hardcoded, so this stays correct even if `name_prefix` or the AWS account/region changes.

In [ ]:
import subprocess
from lab_helpers.aws_clients import REGION, boto_session, s3_client

ecr_client = boto_session.client("ecr")


def terraform_output(name: str) -> str:
    """Read a single output value from the Terraform state in ../../iac."""
    result = subprocess.run(
        ["terraform", "-chdir=../../iac", "output", "-raw", name],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()


artifacts_bucket = terraform_output("artifacts_bucket_id")
kb_data_bucket = terraform_output("kb_data_bucket_id")
ecr_repository_name = terraform_output("ecr_repository_name")

print("✅ Resources located:")
print(f"  🪣 Artifacts bucket: {artifacts_bucket}")
print(f"  🪣 KB data bucket: {kb_data_bucket}")
print(f"  📦 ECR repository: {ecr_repository_name}")
print(f"  🌍 Region: {REGION}")

## Step 2: Empty the S3 Buckets

The `artifacts` bucket has versioning enabled, so deleting just the current object versions isn't enough — every noncurrent version and delete marker has to go too, or `terraform destroy` fails with `BucketNotEmpty`.

In [ ]:
def empty_bucket(bucket_name: str) -> None:
    paginator = s3_client.get_paginator("list_object_versions")
    deleted = 0
    for page in paginator.paginate(Bucket=bucket_name):
        objects_to_delete = [
            {"Key": v["Key"], "VersionId": v["VersionId"]}
            for v in page.get("Versions", []) + page.get("DeleteMarkers", [])
        ]
        if objects_to_delete:
            s3_client.delete_objects(Bucket=bucket_name, Delete={"Objects": objects_to_delete})
            deleted += len(objects_to_delete)
    print(f"  🗑️  Deleted {deleted} object version(s) from {bucket_name}")


print("🪣 Emptying S3 buckets...")
empty_bucket(artifacts_bucket)
empty_bucket(kb_data_bucket)
print("✅ S3 buckets emptied")

## Step 3: Delete Pushed Container Images from ECR

The repository has no `force_delete` set, so `terraform destroy` would fail on any image still pushed there (every image built and pushed in Lab 4).

In [ ]:
print("📦 Deleting images from ECR repository...")
image_ids = ecr_client.list_images(repositoryName=ecr_repository_name)["imageIds"]

if image_ids:
    ecr_client.batch_delete_image(repositoryName=ecr_repository_name, imageIds=image_ids)
    print(f"  🗑️  Deleted {len(image_ids)} image(s) from {ecr_repository_name}")
else:
    print("  ℹ️  No images found — repository is already empty")

print("✅ ECR repository emptied")

## Step 4: Destroy All Terraform-Managed Infrastructure

This single command removes everything: IAM roles, ECR, S3, DynamoDB, Lambda, both Cognito pools (production `modules/cognito` and the lab `MCPServerPool`), the AgentCore Gateway/Policy Engine, Knowledge Base, Memory, Runtime, Online Evaluation, and CloudWatch log delivery.

Run it from a terminal, not from this notebook — `terraform destroy` prompts for an interactive `yes` confirmation that doesn't work well inside a Jupyter cell:

```bash
cd ../../iac
terraform destroy
```

Review the plan it prints (it should list every resource these labs created), then type `yes` to confirm.

⚠️ **This is irreversible.** It deletes the agent's Memory (conversation history) and the Knowledge Base along with everything else.

Once it finishes, come back and run the verification step below.

## Step 5: Verify Cleanup

After `terraform destroy` finishes, confirm the stack is actually gone — every `aws_ssm_parameter` resource Terraform created should no longer resolve.

In [ ]:
from botocore.exceptions import ClientError
from lab_helpers.utils import get_ssm_parameter

parameters_to_check = [
    "/app/customersupport/agentcore/runtime_arn",
    "/app/customersupport/agentcore/memory_id",
    "/app/customersupport/agentcore/gateway_id",
    "/app/customersupport/agentcore/mcp_lab/pool_id",
]

still_present = []
for name in parameters_to_check:
    try:
        get_ssm_parameter(name, with_decryption=False)
        still_present.append(name)
    except ClientError as e:
        if e.response["Error"]["Code"] != "ParameterNotFound":
            raise

print("=" * 60)
if still_present:
    print("⚠️  CLEANUP INCOMPLETE")
    print("=" * 60)
    print("\nThese parameters still resolve — did `terraform destroy` finish successfully?\n")
    for name in still_present:
        print(f"  - {name}")
else:
    print("🧹 CLEANUP COMPLETED SUCCESSFULLY! 🧹")
    print("=" * 60)
    print("\n✨ Your AWS account is clean and ready for new experiments!")
    print("\nThank you for completing the AgentCore End-to-End tutorial! 🚀")